# Cyclic Sort

A niche but elegant pattern for arrays whose values are a **known contiguous range** — `1..n` or `0..n-1`. Because every value has a unique "home" index, you can sort the array in **O(n) time and O(1) space** by swapping each value home — and the classic "find the missing / duplicate number" problems then fall right out.

> signal → template → worked examples → toolkit

**Contents**

1. **The signal**
2. **The cyclic sort**
3. **Missing & duplicate numbers**
4. **The `0..n` variant**
5. **When to use & cheat-sheet**

## 1. The signal — values are a contiguous range

The trigger is specific: the array holds **n numbers drawn from `1..n`** (or `0..n-1`), possibly with a few missing or duplicated, and the problem asks you to sort it or to find the **missing**, **duplicated**, or **misplaced** values.

The key observation: when the values *are* the index range, each value knows exactly where it belongs — value `v` belongs at index `v-1` (for `1..n`). So you can place everything in **O(n)** with no comparisons and **no extra memory** — beating a general sort's O(n log n) precisely by exploiting the range.

## 2. The cyclic sort

Walk the array; for each position, if the value isn't already home, **swap it to its home index**. Don't advance until the current slot holds the value that belongs there. Each swap puts at least one value in its final place, so there are at most n swaps total — **O(n)** time, **O(1)** space.

In [1]:
def cyclic_sort(nums):
    i = 0
    while i < len(nums):
        home = nums[i] - 1               # value v belongs at index v-1  (range 1..n)
        if nums[i] != nums[home]:
            nums[i], nums[home] = nums[home], nums[i]   # swap it home
        else:
            i += 1                       # already placed -> move on
    return nums

print("cyclic_sort([3,1,5,4,2]):", cyclic_sort([3, 1, 5, 4, 2]))


cyclic_sort([3,1,5,4,2]): [1, 2, 3, 4, 5]


The `while` (not `for`) matters: after a swap, the value that *just arrived* at `i` may also be out of place, so you re-check the same index until it settles. That's also why it stays O(n) — each swap finalizes at least one element.

## 3. Missing & duplicate numbers

Once every value is at its home, **any index `i` whose value isn't `i+1` is evidence**: that slot's rightful value is *missing*, and the value parked there is a *duplicate* (or otherwise out of place). One cyclic-sort pass plus one scan finds **all** missing or duplicated numbers — still O(n) / O(1):

In [2]:
def _place(nums):                        # the cyclic-sort core, reused
    i = 0
    while i < len(nums):
        home = nums[i] - 1
        if nums[i] != nums[home]:
            nums[i], nums[home] = nums[home], nums[i]
        else:
            i += 1

def find_missing(nums):                  # values in 1..n, some duplicated, some missing
    _place(nums)
    return [i + 1 for i, v in enumerate(nums) if v != i + 1]

def find_duplicates(nums):
    _place(nums)
    return [v for i, v in enumerate(nums) if v != i + 1]

print("find_missing([4,3,2,7,8,2,3,1])   :", find_missing([4, 3, 2, 7, 8, 2, 3, 1]))
print("find_duplicates([4,3,2,7,8,2,3,1]):", find_duplicates([4, 3, 2, 7, 8, 2, 3, 1]))


find_missing([4,3,2,7,8,2,3,1])   : [5, 6]
find_duplicates([4,3,2,7,8,2,3,1]): [3, 2]


## 4. The `0..n` variant

When the range is `0..n-1`, the home of value `v` is just index `v` (no `-1`). And the famous **"missing number"** problem gives you `n` distinct values from `0..n` in an array of length `n` — one is missing. Place each value at `index = value` (guarding the single out-of-range value), then the first index whose value ≠ index is the answer:

In [3]:
def missing_number(nums):                # n distinct values from 0..n, one missing
    i, n = 0, len(nums)
    while i < n:
        home = nums[i]
        if nums[i] < n and nums[i] != nums[home]:
            nums[i], nums[home] = nums[home], nums[i]
        else:
            i += 1
    for i in range(n):
        if nums[i] != i:
            return i
    return n                             # 0..n-1 all present -> n is the missing one

print("missing_number([3,0,1]):", missing_number([3, 0, 1]))
print("missing_number([0,1])  :", missing_number([0, 1]))


missing_number([3,0,1]): 2
missing_number([0,1])  : 2


For *just* the single missing number, the XOR trick (`xor of 0..n` against `xor of the array`) or the sum formula (`n(n+1)//2 - sum(nums)`) is even simpler and doesn't mutate the array. Cyclic sort earns its keep when you need to find **multiple** missing/duplicate values, or the full sorted order, in place.

## 5. When to use & cheat-sheet

| The problem says… | Use cyclic sort to… |
|---|---|
| "array of `1..n` (or `0..n-1`)" + sort | place each value home in O(n) |
| "find all numbers that disappeared" | scan for index where value ≠ index+1 |
| "find all duplicates" | the value parked at a wrong index is the dup |
| "find the first missing positive" | same idea, ignore values outside `1..n` |
| "find the corrupt pair (dup + missing)" | both fall out of one pass |

**The recognition cue:** the values are (almost) a **permutation of a known index range** — which lets you use the array itself as the hash table (index = value) for O(1) extra space.

**Reach for something else when:** the values aren't a contiguous range (use a `set` / `Counter`), you only need one missing number (XOR or the sum formula), or you can't mutate the input.

| Task | Time | Space |
|---|:---:|:---:|
| Cyclic sort | O(n) | O(1) |
| Find all missing / duplicates | O(n) | O(1) |
| Single missing via XOR / sum | O(n) | O(1), no mutation |
| General sort *(for contrast)* | O(n log n) | — |